# 04 — Reviewer #10.2: Frozen vs Last-Block Fine-Tuning

**Purpose:** answer the reviewer request for an additional controlled fine-tuning experiment using **MobileNetV2** and **DenseNet201** on the **final leakage-controlled split**.

This notebook runs four experiments:

1. MobileNetV2 — fully frozen convolutional base
2. MobileNetV2 — last convolutional block unfrozen
3. DenseNet201 — fully frozen convolutional base
4. DenseNet201 — last convolutional block unfrozen

All four use the same controlled schedule:

- final clean dataset
- seed = 42
- 224×224 RGB
- batch size = 32
- `/255` rescaling
- horizontal + vertical flips for training only
- Adam, learning rate = 1e-4
- categorical cross-entropy
- 20 epochs
- validation accuracy selects the best checkpoint
- same classification head within each architecture
- untouched final Test split for reporting

Final clean dataset:
- Train = 8,845
- Validation = 2,178
- Test = 2,587
- Test Cataract = 854
- Test Normal = 977
- Test Not Eye = 756

**Do not change the settings while running this reviewer experiment.**

In [ ]:
# ============================================================
# CELL 1 — SETUP, GOOGLE DRIVE, GPU, PATHS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import gc
import json
import time
import random
import platform
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    CSVLogger,
    TerminateOnNaN
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    roc_auc_score
)

SEED = 42
EPOCHS = 20
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
LEARNING_RATE = 1e-4

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT = Path('/content/drive/MyDrive/Cataract')

DATA = PROJECT / 'Data_Clean_LeakageControlled_FINAL'
TRAIN_DIR = DATA / 'Train'
VAL_DIR = DATA / 'Validation'
TEST_DIR = DATA / 'Test'

OUT_ROOT = (
    PROJECT
    / 'FINAL_REVISION_2026_08'
    / 'reviewer_10_2_clean_split'
)

OUT_ROOT.mkdir(parents=True, exist_ok=True)

CLASS_ORDER = ['Cataract', 'Normal', 'Not Eye']

for p in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    assert p.exists(), f'STOP: Missing folder: {p}'

gpus = tf.config.list_physical_devices('GPU')

print('TensorFlow:', tf.__version__)
print('Python:', platform.python_version())
print('GPU devices:', gpus)
print('Dataset:', DATA)
print('Output:', OUT_ROOT)

if not gpus:
    raise RuntimeError(
        'STOP: No GPU detected. In Colab choose '
        'Runtime → Change runtime type → T4 GPU, then rerun CELL 1.'
    )

print('\n✅ CELL 1 COMPLETE')

In [ ]:
# ============================================================
# CELL 2 — DATA GENERATORS + FINAL COUNT CHECK
# ============================================================

def make_generators(seed=SEED):

    train_aug = ImageDataGenerator(
        rescale=1./255,
        horizontal_flip=True,
        vertical_flip=True
    )

    plain = ImageDataGenerator(
        rescale=1./255
    )

    train = train_aug.flow_from_directory(
        TRAIN_DIR,
        target_size=IMAGE_SIZE,
        color_mode='rgb',
        class_mode='categorical',
        classes=CLASS_ORDER,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=seed,
        interpolation='nearest'
    )

    val = plain.flow_from_directory(
        VAL_DIR,
        target_size=IMAGE_SIZE,
        color_mode='rgb',
        class_mode='categorical',
        classes=CLASS_ORDER,
        batch_size=BATCH_SIZE,
        shuffle=False,
        interpolation='nearest'
    )

    test = plain.flow_from_directory(
        TEST_DIR,
        target_size=IMAGE_SIZE,
        color_mode='rgb',
        class_mode='categorical',
        classes=CLASS_ORDER,
        batch_size=BATCH_SIZE,
        shuffle=False,
        interpolation='nearest'
    )

    return train, val, test


train, val, test = make_generators()

print('\nClass indices:', train.class_indices)
print('Train images:', train.samples)
print('Validation images:', val.samples)
print('Test images:', test.samples)

assert train.samples == 8845
assert val.samples == 2178
assert test.samples == 2587

assert train.class_indices == {
    'Cataract': 0,
    'Normal': 1,
    'Not Eye': 2
}

print('\n✅ FINAL CLEAN DATA COUNTS VERIFIED')
print('✅ CELL 2 COMPLETE')

## Fine-tuning definition used in this controlled experiment

For a clean reviewer comparison:

### Frozen baseline
The complete ImageNet convolutional base is frozen. Only the classification head is trainable.

### Last-block fine-tuning
The same model and same head are used, but only the **final convolutional block** of the backbone is made trainable:

- **MobileNetV2:** `block_16_*`, `Conv_1`, and the final backbone batch-normalization layer
- **DenseNet201:** `conv5_block*` plus the final backbone normalization layer

All earlier backbone layers remain frozen.

This makes the comparison controlled: the only experimental change is whether the final backbone block can update.

In [ ]:
# ============================================================
# CELL 3 — MODEL BUILDERS
# ============================================================

def reset_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)


def build_head(base, architecture):

    x = base.output

    # Use the recovered head form used in the main experiments.
    x = layers.Conv2D(
        32,
        (3, 3),
        activation='relu'
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.MaxPooling2D(
        (2, 2)
    )(x)

    x = layers.Dropout(0.17)(x)

    x = layers.Conv2D(
        64,
        (2, 2),
        activation='relu'
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        64,
        activation='relu'
    )(x)

    x = layers.Dense(
        32,
        activation='relu'
    )(x)

    x = layers.Dense(
        32,
        activation='relu'
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Dropout(0.30)(x)

    output = layers.Dense(
        3,
        activation='softmax',
        name='preds'
    )(x)

    return Model(
        inputs=base.input,
        outputs=output
    )


def set_backbone_trainability(
    base,
    architecture,
    mode
):

    # Start from a completely frozen backbone.
    for layer in base.layers:
        layer.trainable = False

    if mode == 'frozen':
        return

    if mode != 'last_block':
        raise ValueError(
            "mode must be 'frozen' or 'last_block'"
        )

    if architecture == 'MobileNetV2':

        for layer in base.layers:

            if (
                layer.name.startswith('block_16_')
                or layer.name == 'Conv_1'
                or layer.name == 'Conv_1_bn'
            ):
                layer.trainable = True

    elif architecture == 'DenseNet201':

        for layer in base.layers:

            if (
                layer.name.startswith('conv5_block')
                or layer.name == 'bn'
            ):
                layer.trainable = True

    else:
        raise ValueError(
            f'Unsupported architecture: {architecture}'
        )


def build_model(
    architecture,
    mode
):

    tf.keras.backend.clear_session()
    gc.collect()
    reset_seed()

    if architecture == 'MobileNetV2':

        base = tf.keras.applications.MobileNetV2(
            weights='imagenet',
            include_top=False,
            input_shape=(224, 224, 3)
        )

    elif architecture == 'DenseNet201':

        base = tf.keras.applications.DenseNet201(
            weights='imagenet',
            include_top=False,
            input_shape=(224, 224, 3)
        )

    else:
        raise ValueError(architecture)

    set_backbone_trainability(
        base,
        architecture,
        mode
    )

    model = build_head(
        base,
        architecture
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model, base


print('✅ Model builders ready')
print('✅ CELL 3 COMPLETE')

In [ ]:
# ============================================================
# CELL 4 — SANITY-CHECK TRAINABLE LAYERS
# ============================================================

for architecture in [
    'MobileNetV2',
    'DenseNet201'
]:

    for mode in [
        'frozen',
        'last_block'
    ]:

        model, base = build_model(
            architecture,
            mode
        )

        trainable_backbone = [
            layer.name
            for layer in base.layers
            if layer.trainable
        ]

        print('\n====================================')
        print(architecture, '|', mode)
        print('====================================')

        print(
            'Backbone layers:',
            len(base.layers)
        )

        print(
            'Trainable backbone layers:',
            len(trainable_backbone)
        )

        if mode == 'frozen':
            assert len(trainable_backbone) == 0

        if mode == 'last_block':
            assert len(trainable_backbone) > 0

        print(
            'First trainable backbone names:',
            trainable_backbone[:10]
        )

        print(
            'Last trainable backbone names:',
            trainable_backbone[-10:]
        )

        del model, base
        tf.keras.backend.clear_session()
        gc.collect()

print('\n✅ TRAINABILITY SANITY CHECK PASSED')
print('✅ CELL 4 COMPLETE')

In [ ]:
# ============================================================
# CELL 5 — TRAIN ONE CONTROLLED EXPERIMENT FUNCTION
# ============================================================

def run_experiment(
    architecture,
    mode
):

    run_name = f'{architecture}_{mode}'

    run_dir = OUT_ROOT / run_name
    run_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    done = run_dir / 'DONE.txt'

    if done.exists():
        print(
            f'✅ SKIP: {run_name} already complete'
        )

        summary_file = (
            run_dir
            / 'summary.json'
        )

        if summary_file.exists():
            with open(summary_file) as f:
                return json.load(f)

        return None

    print('\n========================================')
    print('STARTING:', run_name)
    print('========================================')

    # Fresh generators for each controlled run
    train_gen, val_gen, test_gen = (
        make_generators(SEED)
    )

    model, base = build_model(
        architecture,
        mode
    )

    trainability = pd.DataFrame([
        {
            'index': i,
            'layer': layer.name,
            'class': layer.__class__.__name__,
            'trainable': layer.trainable,
            'in_backbone': layer.name in {
                x.name for x in base.layers
            }
        }
        for i, layer in enumerate(model.layers)
    ])

    trainability.to_csv(
        run_dir
        / 'layer_trainability.csv',
        index=False
    )

    best_path = (
        run_dir
        / 'best.keras'
    )

    callbacks = [
        ModelCheckpoint(
            best_path,
            monitor='val_accuracy',
            mode='max',
            save_best_only=True,
            verbose=1
        ),
        CSVLogger(
            run_dir
            / 'history.csv'
        ),
        TerminateOnNaN()
    ]

    t0 = time.time()

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    training_seconds = (
        time.time() - t0
    )

    (
        run_dir
        / 'training_time_seconds.txt'
    ).write_text(
        str(training_seconds)
    )

    # Evaluate only the validation-selected checkpoint.
    best_model = tf.keras.models.load_model(
        best_path,
        compile=False
    )

    test_gen.reset()

    probs = best_model.predict(
        test_gen,
        verbose=1
    )

    y_true = test_gen.classes.copy()
    y_pred = probs.argmax(axis=1)

    np.save(
        run_dir / 'probs.npy',
        probs
    )

    np.save(
        run_dir / 'y_true.npy',
        y_true
    )

    np.save(
        run_dir / 'y_pred.npy',
        y_pred
    )

    pd.DataFrame({
        'filepath': test_gen.filepaths,
        'y_true': y_true,
        'y_pred': y_pred
    }).to_csv(
        run_dir
        / 'test_predictions_index.csv',
        index=False
    )

    # 3-class test accuracy
    three_acc = accuracy_score(
        y_true,
        y_pred
    )

    # Clinical Cataract-v-Normal subset
    clinical_mask = np.isin(
        y_true,
        [0, 1]
    )

    clinical_true3 = (
        y_true[clinical_mask]
    )

    clinical_probs = (
        probs[clinical_mask]
    )

    y_binary = (
        clinical_true3 == 0
    ).astype(int)

    denominator = (
        clinical_probs[:, 0]
        + clinical_probs[:, 1]
    )

    cataract_score = np.divide(
        clinical_probs[:, 0],
        denominator,
        out=np.full(
            denominator.shape,
            0.5,
            dtype=float
        ),
        where=denominator > 0
    )

    binary_pred = (
        cataract_score >= 0.5
    ).astype(int)

    TP = int(
        (
            (y_binary == 1)
            &
            (binary_pred == 1)
        ).sum()
    )

    FN = int(
        (
            (y_binary == 1)
            &
            (binary_pred == 0)
        ).sum()
    )

    TN = int(
        (
            (y_binary == 0)
            &
            (binary_pred == 0)
        ).sum()
    )

    FP = int(
        (
            (y_binary == 0)
            &
            (binary_pred == 1)
        ).sum()
    )

    clinical_accuracy = (
        TP + TN
    ) / len(y_binary)

    sensitivity = (
        TP
        / (TP + FN)
    )

    specificity = (
        TN
        / (TN + FP)
    )

    auc_value = roc_auc_score(
        y_binary,
        cataract_score
    )

    history_df = pd.read_csv(
        run_dir
        / 'history.csv'
    )

    best_epoch_idx = int(
        history_df[
            'val_accuracy'
        ].astype(float).idxmax()
    )

    summary = {
        'architecture': architecture,
        'mode': mode,
        'seed': SEED,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'train_n': int(train_gen.samples),
        'validation_n': int(val_gen.samples),
        'test_n': int(test_gen.samples),
        'best_epoch': best_epoch_idx + 1,
        'best_validation_accuracy': float(
            history_df.loc[
                best_epoch_idx,
                'val_accuracy'
            ]
        ),
        'training_seconds': float(
            training_seconds
        ),
        'test_3class_accuracy': float(
            three_acc
        ),
        'clinical_n': int(
            len(y_binary)
        ),
        'clinical_accuracy': float(
            clinical_accuracy
        ),
        'sensitivity': float(
            sensitivity
        ),
        'specificity': float(
            specificity
        ),
        'auc': float(
            auc_value
        ),
        'TP': TP,
        'FN': FN,
        'TN': TN,
        'FP': FP
    }

    with open(
        run_dir
        / 'summary.json',
        'w'
    ) as f:
        json.dump(
            summary,
            f,
            indent=2
        )

    done.write_text(
        'completed\n'
        f'architecture={architecture}\n'
        f'mode={mode}\n'
        f'seed={SEED}\n'
        f'epochs={EPOCHS}\n'
    )

    print('\n✅ COMPLETE:', run_name)
    print(json.dumps(summary, indent=2))

    del (
        model,
        base,
        best_model,
        train_gen,
        val_gen,
        test_gen
    )

    tf.keras.backend.clear_session()
    gc.collect()

    return summary


print('✅ Experiment runner ready')
print('✅ CELL 5 COMPLETE')

## Run the four experiments

You may run the next four cells one after another.

Each completed experiment writes `DONE.txt`, so if Colab disconnects you can rerun the notebook and completed experiments will be skipped.

In [ ]:
# ============================================================
# CELL 6A — MOBILEnetV2 FROZEN BASELINE
# ============================================================

mobile_frozen = run_experiment(
    'MobileNetV2',
    'frozen'
)

print('\n✅ CELL 6A COMPLETE')

In [ ]:
# ============================================================
# CELL 6B — MOBILEnetV2 LAST-BLOCK FINE-TUNING
# ============================================================

mobile_last = run_experiment(
    'MobileNetV2',
    'last_block'
)

print('\n✅ CELL 6B COMPLETE')

In [ ]:
# ============================================================
# CELL 6C — DENSENET201 FROZEN BASELINE
# ============================================================

dense_frozen = run_experiment(
    'DenseNet201',
    'frozen'
)

print('\n✅ CELL 6C COMPLETE')

In [ ]:
# ============================================================
# CELL 6D — DENSENET201 LAST-BLOCK FINE-TUNING
# ============================================================

dense_last = run_experiment(
    'DenseNet201',
    'last_block'
)

print('\n✅ CELL 6D COMPLETE')

In [ ]:
# ============================================================
# CELL 7 — BUILD FINAL REVIEWER #10.2 COMPARISON TABLE
# ============================================================

rows = []

for architecture in [
    'MobileNetV2',
    'DenseNet201'
]:

    for mode in [
        'frozen',
        'last_block'
    ]:

        run_dir = (
            OUT_ROOT
            / f'{architecture}_{mode}'
        )

        summary_path = (
            run_dir
            / 'summary.json'
        )

        if not summary_path.exists():
            raise RuntimeError(
                f'STOP: Missing {summary_path}'
            )

        with open(summary_path) as f:
            s = json.load(f)

        rows.append({
            'Model': architecture,
            'Condition': (
                'Frozen backbone'
                if mode == 'frozen'
                else 'Last block unfrozen'
            ),
            'Best Epoch': s['best_epoch'],
            'Validation Accuracy (%)':
                100 * s[
                    'best_validation_accuracy'
                ],
            '3-Class Test Accuracy (%)':
                100 * s[
                    'test_3class_accuracy'
                ],
            'Clinical Accuracy (%)':
                100 * s[
                    'clinical_accuracy'
                ],
            'Sensitivity (%)':
                100 * s[
                    'sensitivity'
                ],
            'Specificity (%)':
                100 * s[
                    'specificity'
                ],
            'Clinical AUC':
                s['auc'],
            'Training Time (min)':
                s[
                    'training_seconds'
                ] / 60,
            'TP': s['TP'],
            'FN': s['FN'],
            'TN': s['TN'],
            'FP': s['FP']
        })


comparison = pd.DataFrame(rows)

comparison.to_csv(
    OUT_ROOT
    / 'Reviewer_10_2_Final_Comparison.csv',
    index=False
)

print('\n========================================')
print('REVIEWER #10.2 FINAL COMPARISON')
print('========================================')

display(
    comparison.round(4)
)

print('\n✅ CELL 7 COMPLETE')

In [ ]:
# ============================================================
# CELL 8 — CALCULATE IMPROVEMENT FROM LAST-BLOCK FINE-TUNING
# ============================================================

improvement_rows = []

for model_name in [
    'MobileNetV2',
    'DenseNet201'
]:

    frozen = comparison[
        (
            comparison['Model']
            == model_name
        )
        &
        (
            comparison['Condition']
            == 'Frozen backbone'
        )
    ].iloc[0]

    tuned = comparison[
        (
            comparison['Model']
            == model_name
        )
        &
        (
            comparison['Condition']
            == 'Last block unfrozen'
        )
    ].iloc[0]

    improvement_rows.append({
        'Model': model_name,
        'Delta Validation Accuracy (pp)':
            tuned[
                'Validation Accuracy (%)'
            ]
            -
            frozen[
                'Validation Accuracy (%)'
            ],
        'Delta 3-Class Test Accuracy (pp)':
            tuned[
                '3-Class Test Accuracy (%)'
            ]
            -
            frozen[
                '3-Class Test Accuracy (%)'
            ],
        'Delta Clinical Accuracy (pp)':
            tuned[
                'Clinical Accuracy (%)'
            ]
            -
            frozen[
                'Clinical Accuracy (%)'
            ],
        'Delta Sensitivity (pp)':
            tuned[
                'Sensitivity (%)'
            ]
            -
            frozen[
                'Sensitivity (%)'
            ],
        'Delta Specificity (pp)':
            tuned[
                'Specificity (%)'
            ]
            -
            frozen[
                'Specificity (%)'
            ],
        'Delta AUC':
            tuned[
                'Clinical AUC'
            ]
            -
            frozen[
                'Clinical AUC'
            ]
    })


improvement = pd.DataFrame(
    improvement_rows
)

improvement.to_csv(
    OUT_ROOT
    / 'Reviewer_10_2_Improvement.csv',
    index=False
)

print('\n========================================')
print('LAST-BLOCK IMPROVEMENT OVER FROZEN')
print('========================================')

display(
    improvement.round(4)
)

print('\n✅ CELL 8 COMPLETE')

In [ ]:
# ============================================================
# CELL 9 — FINAL COMPLETION CHECK
# ============================================================

required_runs = [
    'MobileNetV2_frozen',
    'MobileNetV2_last_block',
    'DenseNet201_frozen',
    'DenseNet201_last_block'
]

for run_name in required_runs:

    run_dir = OUT_ROOT / run_name

    required = [
        'best.keras',
        'history.csv',
        'training_time_seconds.txt',
        'layer_trainability.csv',
        'probs.npy',
        'y_true.npy',
        'y_pred.npy',
        'test_predictions_index.csv',
        'summary.json',
        'DONE.txt'
    ]

    for fn in required:

        p = run_dir / fn

        if not p.exists():
            raise RuntimeError(
                f'STOP: {run_name} missing {fn}'
            )


for fn in [
    'Reviewer_10_2_Final_Comparison.csv',
    'Reviewer_10_2_Improvement.csv'
]:

    if not (
        OUT_ROOT / fn
    ).exists():

        raise RuntimeError(
            f'STOP: Missing {fn}'
        )


(
    OUT_ROOT
    / 'REVIEWER_10_2_DONE.txt'
).write_text(
    'Reviewer #10.2 controlled clean-split '
    'fine-tuning experiment completed.\n'
    'Models: MobileNetV2 and DenseNet201\n'
    'Conditions: frozen backbone vs last block unfrozen\n'
    'Seed: 42\n'
    'Epochs: 20\n'
)


print('========================================')
print('✅ REVIEWER #10.2 EXPERIMENT COMPLETE')
print('========================================')

print('\nResults saved to:')
print(OUT_ROOT)

print(
    '\nNext: download/save this executed notebook '
    'and upload it back to ChatGPT for interpretation.'
)